# Initialize the db

### imports:
* os - get the env data
* sqlalchemy - ORM manage to work with sql with python
* psycopg2 - Adapter between postgres and python


### Create Engine and connceting to it
* provides services for execution of SQL statements as well as transaction control

In [1]:
try:
    import os
    import pandas as pd
    from sqlalchemy import create_engine, text
    from dotenv import load_dotenv
    from datetime import datetime, timedelta
    print("Success Importing")
except ImportError as e:
    print(f"Some Imports Failed: {e}")

#loading data from the .env file
load_dotenv()

#using os to get the data from env
USER = os.getenv('DB_USER')
PASSWORD = os.getenv('DB_PASSWORD')
HOST = os.getenv('DB_HOST')
PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

connection_string = f"postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

try:
    engine = create_engine(connection_string)
    with engine.connect() as connection:
        print(f"Success connection to the db: {DB_NAME}")
except Exception as e:
    print(f"Failed to connect to db: {e}")

Success Importing
Success connection to the db: postgres


## Creating The Tables
* creating the sql Tables
* open connection to the db to execute the create table SQL func using .execute(text())
* using commit to commit the changes to the db

In [2]:
create_tables_sql = """
CREATE TABLE IF NOT EXISTS suppliers (
    supplier_id INT PRIMARY KEY,
    supplier_name VARCHAR(100),
    country VARCHAR(50)
);

CREATE TABLE IF NOT EXISTS parts (
    part_id INT PRIMARY KEY,
    part_name VARCHAR(100),
    category VARCHAR(50),
    unit_cost DECIMAL(10, 2)
);

CREATE TABLE IF NOT EXISTS purchase_orders (
    po_id INT PRIMARY KEY,
    supplier_id INT REFERENCES suppliers(supplier_id),
    part_id INT REFERENCES parts(part_id),
    quantity INT,
    order_date TIMESTAMP,
    expected_arrival TIMESTAMP,
    actual_arrival TIMESTAMP
);

CREATE TABLE IF NOT EXISTS customer_deliveries (
    delivery_id INT PRIMARY KEY,
    part_id INT REFERENCES parts(part_id),
    customer_name VARCHAR(100),
    quantity INT,
    delivery_date TIMESTAMP,
    status VARCHAR(20)
);
"""

try:
    with engine.connect() as connection:
        connection.execute(text(create_tables_sql))
        connection.commit()
    print("The Tables Created!")
except Exception as e:
    print(f"Error Table Not Created: {e}")

The Tables Created!


### Injecting some data to the tables
* injecting using .to_sql(name, engine, ...)

In [3]:
df_suppliers = pd.DataFrame({
    'supplier_id': [101, 102, 103],
    'supplier_name': ['TechParts Germany', 'AsianLogistics', 'IsraelPrecision'],
    'country': ['Germany', 'China', 'Israel']
})

df_parts = pd.DataFrame({
    'part_id': [1, 2, 3, 4],
    'part_name': ['Motherboard V2', 'Hydraulic Pump', 'Sensor Unit', 'Steel Frame'],
    'category': ['Electronics', 'Mechanical', 'Electronics', 'Structure'],
    'unit_cost': [500, 1200, 150, 3000]
})

today = datetime.now()

df_purchase_orders = pd.DataFrame({
    'po_id': [5001, 5002, 5003, 5004],
    'supplier_id': [101, 102, 101, 103],
    'part_id': [1, 2, 3, 1],
    'quantity': [50, 20, 100, 40],
    'order_date': [today - timedelta(days=20)] * 4,
    
    'expected_arrival': [
        today - timedelta(days=10),
        today - timedelta(days=10),
        today - timedelta(days=10),
        today + timedelta(days=5)
    ],
    
    'actual_arrival': [
        today - timedelta(days=2),
        today - timedelta(days=9),
        today - timedelta(days=10),
        None
    ]
})

df_customer_deliveries = pd.DataFrame({
    'delivery_id': [9001, 9002, 9003, 9004],
    'part_id': [1, 4, 1, 2],
    'customer_name': ['Intel', 'Tesla', 'Apple', 'SpaceX'],
    'quantity': [150, 50, 200, 30],
    'delivery_date': [today - timedelta(days=1), today + timedelta(days=10), today - timedelta(days=2), today + timedelta(days=5)],
    'status': ['Shipped', 'Pending', 'Delivered', 'Shipped']
})

try:
    df_suppliers.to_sql('suppliers', engine, if_exists='append', index=False)
    df_parts.to_sql('parts', engine, if_exists='append', index=False)
    df_purchase_orders.to_sql('purchase_orders', engine, if_exists='append', index=False)
    df_customer_deliveries.to_sql('customer_deliveries', engine, if_exists='append', index=False)
    print("Success injected to the tables!")
except Exception as e:
    print(f"Error injected to the tables: {e}")

Success injected to the tables!
